# Sprzątanie po warsztacie (prowadzący)

**Kto:** prowadzący, na workspace Premium (i opcjonalnie na koncie testowym Free).

Endpoint AI Search, aplikacja i Knowledge Assistant kosztują także wtedy, gdy nikt z nich nie korzysta. Notebook domyślnie działa w trybie **`DRY_RUN = True`**: tylko wypisuje, co by usunął. Przełącz flagę dopiero po przeczytaniu listy.

| Zasób | Domyślnie | Flaga |
|---|---|---|
| Databricks App `agent-sqlday-retail-agent` oraz `agent-sqlday-retail-agent-live` | usuń | — |
| model `workspace.default.retail_customer_agent` razem z wersjami | usuń | — |
| indeks i endpoint AI Search | usuń | — |
| Genie Agent `Retail Customer Intelligence Assistant` | do kosza | — |
| funkcje UC, tabele warsztatu, Volume z raportami, schematy `bakehouse`, `airbnb` i `governance` | zostaw | `DROP_DATA = True` |
| Knowledge Assistant (retail i robotyka) | usuń | — |

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
dbutils.library.restartPython()

In [ ]:
# Wspólna konfiguracja warsztatu. Ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
BH_SCHEMA = "bakehouse"       # ścieżka B: kopie danych Bakehouse i funkcje-narzędzia
AIRBNB_SCHEMA = "airbnb"      # ścieżka C: oferty Airbnb i funkcje-narzędzia
POLICY_SCHEMA = "governance"  # maski i filtry ścieżek B i C: poza schematami, które MCP wystawia agentowi
BH_TRANSACTIONS = f"{CATALOG}.{BH_SCHEMA}.transactions"
BH_REVIEWS = f"{CATALOG}.{BH_SCHEMA}.reviews"
AIRBNB_TABLE = f"{CATALOG}.{AIRBNB_SCHEMA}.listings"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

import logging
# MLflow w notebooku serverless (UI) wypisuje przy tracingu stos Py4JSecurityException z "resolving tags".
# To ostrzeżenie, nie błąd. Trace zapisuje się poprawnie, a wyciszamy je, żeby nikt nie wziął go za błąd.
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

In [ ]:
from databricks.ai_search.client import AISearchClient
from databricks.sdk import WorkspaceClient

DRY_RUN = True     # False: naprawdę usuń
DROP_DATA = False  # True: usuń także tabele, funkcje i Volume warsztatu

w = WorkspaceClient()
search_client = AISearchClient(disable_notice=True)
APP_NAME = "agent-sqlday-retail-agent"          # interfejs Export wymusza prefiks agent-
LIVE_APP_NAME = "agent-sqlday-retail-agent-live"  # eksport na żywo z M5, kroki 9a–9i
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.retail_customer_agent"
# Genie Agenty warsztatu: TechRetail (M4 · A), Bakehouse (M4 · B), Airbnb (M4 · C)
GENIE_TITLES = (GENIE_TITLE, "Bakehouse Sales Assistant", "Airbnb Listings Assistant")
try:
    genie_spaces = [(s.space_id, s.title) for s in (w.genie.list_spaces().spaces or []) if s.title in GENIE_TITLES]
except Exception as e:
    genie_spaces = []
    print(f"ℹ️  Genie API niedostępne: {type(e).__name__}: {str(e)[:120]}")
# Knowledge Assistanty warsztatu: retail z M3 i robotyka z p3 (REST, bo SDK przypięte przez unitycatalog-ai go nie zna)
KA_NAMES = ("Retail Customer Knowledge Assistant", "retail-customer-knowledge-assistant", "robotics-course-specialist")
try:
    ka_ids = [(ka["name"], ka["display_name"]) for ka in w.api_client.do("GET", "/api/2.1/knowledge-assistants").get("knowledge_assistants", [])
              if ka.get("display_name") in KA_NAMES]
except Exception as e:  # na Free Edition tego API może nie być
    ka_ids = []
    print(f"ℹ️  Knowledge Assistants niedostępne: {type(e).__name__}: {str(e)[:120]}")


def delete_model(full_name):
    # Model z wersjami nie da się usunąć jednym wywołaniem: najpierw wersje, potem model.
    for version in w.model_versions.list(full_name=full_name):
        w.model_versions.delete(full_name=full_name, version=version.version)
    w.registered_models.delete(full_name=full_name)


actions = [
    (f"Databricks App {APP_NAME}", lambda: w.apps.delete(name=APP_NAME)),
    (f"Databricks App {LIVE_APP_NAME}", lambda: w.apps.delete(name=LIVE_APP_NAME)),
    (f"model {UC_MODEL_NAME}", lambda: delete_model(UC_MODEL_NAME)),
    (f"indeks {SEARCH_INDEX}", lambda: search_client.delete_index(SEARCH_ENDPOINT, SEARCH_INDEX)),
    (f"indeks robotyki", lambda: search_client.delete_index(SEARCH_ENDPOINT, f"{CATALOG}.{SCHEMA}.robotics_chunks_index")),
    # capstone zakłada indeks w schemacie swoich danych; ten w default to stary układ sprzed 21.09
    *[(f"indeks capstone {CATALOG}.{schema}.capstone_docs_index",
       lambda schema=schema: search_client.delete_index(SEARCH_ENDPOINT, f"{CATALOG}.{schema}.capstone_docs_index"))
      for schema in (BH_SCHEMA, AIRBNB_SCHEMA, SCHEMA)],
    (f"endpoint AI Search {SEARCH_ENDPOINT}", lambda: search_client.delete_endpoint(SEARCH_ENDPOINT)),
    *[(f"Genie Agent {title} ({space_id})", lambda space_id=space_id: w.genie.trash_space(space_id)) for space_id, title in genie_spaces],
    *[(f"Knowledge Assistant {name}", lambda ka_id=ka_id: w.api_client.do("DELETE", f"/api/2.1/{ka_id}")) for ka_id, name in ka_ids],
]
if DROP_DATA:
    # Najpierw polityki z M4 na tabeli Gold (gdy uczestnik nie doszedł do m4-cleanup), potem ich funkcje.
    actions.append((f"row filter na {GOLD_TABLE}", lambda: spark.sql(f"ALTER TABLE {GOLD_TABLE} DROP ROW FILTER")))
    actions.append((f"maska tax_id na {GOLD_TABLE}", lambda: spark.sql(f"ALTER TABLE {GOLD_TABLE} ALTER COLUMN tax_id DROP MASK")))
    for function in ("retail_row_filter", "mask_tax_id", "get_revenue_summary", "get_average_customer_value", "get_customer_profile", "format_customer_for_agent",
                     "bh_franchise_summary", "bh_product_sales", "bh_payment_methods", "bh_mask_card",
                     "capstone_franchise_sales", "capstone_neighbourhood_summary",
                     # stary układ sprzed 21.09 (ścieżki B i C w default): serwer MCP w M6 wystawiłby je agentowi
                     "get_airbnb_summary", "mask_host_name", "airbnb_row_filter"):
        actions.append((f"funkcja {function}", lambda f=function: spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.{f}")))
    for table in (GOLD_TABLE, DOCS_TABLE, CHUNKS_TABLE, f"{CATALOG}.{SCHEMA}.m1_baseline_answers", f"{CATALOG}.{SCHEMA}.bh_transactions", AIRBNB_TABLE,
                  f"{CATALOG}.{SCHEMA}.capstone_table", f"{CATALOG}.{SCHEMA}.capstone_docs",
                  f"{CATALOG}.{SCHEMA}.robotics_parsed_documents", f"{CATALOG}.{SCHEMA}.robotics_chunks",
                  f"{CATALOG}.{SCHEMA}.airbnb_listings", f"{CATALOG}.{SCHEMA}.bh_transactions"):  # stary układ sprzed 21.09
        actions.append((f"tabela {table}", lambda t=table: spark.sql(f"DROP TABLE IF EXISTS {t}")))
    for volume in (VOLUME, "robotics_files"):
        actions.append((f"Volume {volume}", lambda v=volume: spark.sql(f"DROP VOLUME IF EXISTS {CATALOG}.{SCHEMA}.{v}")))
    # Ścieżki B i C mają własne schematy: jedno DROP SCHEMA ... CASCADE usuwa tabele, funkcje i polityki.
    for schema in (BH_SCHEMA, AIRBNB_SCHEMA, POLICY_SCHEMA):
        actions.append((f"schemat {CATALOG}.{schema}", lambda s=schema: spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{s} CASCADE")))

for label, action in actions:
    if DRY_RUN:
        print(f"[DRY_RUN] usunąłbym: {label}")
        continue
    try:
        action()
        print(f"✅ usunięto: {label}")
    except Exception as e:
        print(f"ℹ️  {label}: {type(e).__name__}: {str(e)[:120]}")

## Koszt dnia (dzień po warsztacie)

Tabela systemowa `system.billing.usage` pokazuje zużycie DBU per produkt z opóźnieniem do kilku godzin. Zapisz wynik w `docs/rehearsal_log.md`: to budżet następnej edycji.

In [ ]:
%sql
SELECT usage_date, billing_origin_product, ROUND(SUM(usage_quantity), 2) AS dbu
FROM system.billing.usage
WHERE usage_date >= date_sub(current_date(), 3)
GROUP BY usage_date, billing_origin_product
ORDER BY usage_date DESC, dbu DESC